In [17]:
import os

In [18]:
# os.system("ros2 node list")
# os.system("ros2 topic list")
# os.system("ros2 service list")

In [33]:
# Home the robot
os.system('ros2 service call /home_the_robot std_srvs/srv/Trigger "{}"')

waiting for service to become available...
requester: making request: std_srvs.srv.Trigger_Request()

response:
std_srvs.srv.Trigger_Response(success=True, message='Homed the robot.')



0

In [34]:

# Stow the robot
os.system('ros2 service call /stow_the_robot std_srvs/srv/Trigger "{}"')

waiting for service to become available...
requester: making request: std_srvs.srv.Trigger_Request()

response:
std_srvs.srv.Trigger_Response(success=True, message='Stowed the robot.')



0

In [21]:
# Stop the robot
os.system('ros2 service call /stop_the_robot std_srvs/srv/Trigger "{}"')

waiting for service to become available...
requester: making request: std_srvs.srv.Trigger_Request()

response:
std_srvs.srv.Trigger_Response(success=True, message='Stopped the robot.')



0

In [35]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist
from std_msgs.msg import Float64MultiArray
import stretch_mujoco.config as config
# import numpy as np
from utils import format_joint_pos

import math
pi = math.pi

try:
    rclpy.init()
except:
    print("rclpy already initialized")

rclpy already initialized


In [36]:
class StretchMujocoPublisher(Node):
    def __init__(self):
        super().__init__('mujoco_publisher')

        self.base_vel_pub = self.create_publisher(Twist, 'cmd_vel', 10)
        self.joint_pose_pub = self.create_publisher(Float64MultiArray, 'joint_pose_cmd', 10)
        self.move_by_pos_pub = self.create_publisher(Float64MultiArray, 'move_by_pos_cmd', 10)
        self.move_manipulator = self.create_publisher(Float64MultiArray, 'move_manipulator_cmd', 10)

    def publish_cmd_vel(self, linear_x=0.0, angular_z=0.0):
        """ Publish Twist message to control velocity """
        msg = Twist()
        msg.linear.x = linear_x
        msg.angular.z = angular_z
        self.base_vel_pub.publish(msg)
        self.get_logger().info(f'Published cmd_vel: linear_x={linear_x}, angular_z={angular_z}')

    def publish_joint_pose(self, positions):
        """ Publish Float64MultiArray for joint positions """
        msg = Float64MultiArray()
        msg.data = positions
        self.joint_pose_pub.publish(msg)
        self.get_logger().info(f'Published joint_pose_cmd: {positions}')

    def publish_move_by_pos(self, positions):
        """ Publish Float64MultiArray for move-by-position command """
        msg = Float64MultiArray()
        msg.data = positions
        self.move_by_pos_pub.publish(msg)
        self.get_logger().info(f'Published move_by_pos_cmd: {positions}')
        
    def publish_move_manipulator(self, positions):
        """ Publish Float64MultiArray for move-by-position command """
        msg = Float64MultiArray()
        msg.data = positions
        self.move_manipulator.publish(msg)
        self.get_logger().info(f'Published move_by_pos_cmd: {positions}')

node = StretchMujocoPublisher()


[WARN] [1740903082.446519509] [rcl.logging_rosout]: Publisher already registered for provided node name. If this is due to multiple nodes with the same name then all logs for that logger name will go out over the existing publisher. As soon as any node with that name is destructed it will unregister the publisher, preventing any further logs for that name from being published on the rosout topic.


In [37]:
mujoco_actuators = ["left_wheel_vel", "right_wheel_vel", "lift", "arm", "wrist_yaw", "wrist_pitch", "wrist_roll",  "gripper", "head_pan", "head_tilt",]
ros_actuators = config.allowed_position_actuators
print("ROS node follows the order of allowed position actuators:\n", ros_actuators)
print("Mujoco followes xml actuators:\n",mujoco_actuators)

ROS node follows the order of allowed position actuators:
 ['arm', 'gripper', 'head_pan', 'head_tilt', 'lift', 'wrist_pitch', 'wrist_roll', 'wrist_yaw', 'base_rotate', 'base_translate']
Mujoco followes xml actuators:
 ['left_wheel_vel', 'right_wheel_vel', 'lift', 'arm', 'wrist_yaw', 'wrist_pitch', 'wrist_roll', 'gripper', 'head_pan', 'head_tilt']


In [38]:
node.publish_cmd_vel(0.0, 0.0)

[INFO] [1740903085.310587922] [mujoco_publisher]: Published cmd_vel: linear_x=0.0, angular_z=0.0


In [43]:
# Test move_by_pos_cmd subscriber
import math
pi = math.pi

arm             = 0
gripper         = 0
head_pan        = 0
head_tilt       = 0
lift            = 0
wrist_pitch     = 0
wrist_roll      = 0
wrist_yaw       = 0
base_rotate     = 0
base_translate  = 1
# base_translate  = 0.5

dpos = [arm, gripper, head_pan, head_tilt, lift, wrist_pitch, wrist_roll, wrist_yaw, base_rotate, base_translate]    # motion following the order of allowed_position_actuators
dpos = [float(item) for item in dpos]
ros_actuators = config.allowed_position_actuators
assert len(dpos) == len(ros_actuators), f"Length of dpos {len(dpos)} does not match ros_actuators {len(ros_actuators)}"
node.publish_move_by_pos(dpos)

[INFO] [1740903134.039604026] [mujoco_publisher]: Published move_by_pos_cmd: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


In [44]:
# Test move_manipulator_cmd topic
import math
pi = math.pi

# <key name="stow" ctrl="0 0 0.23 0 3.14 -0.4 0 0 0 0"/>
stow_mjc_qpos, stow_ros_qpos = format_joint_pos(0, 0, 0.23, 0, 3.14, -0.4, 0, 0, 0, 0)
# <key name="home" ctrl="0 0 0.6 0.1 0 0 0 0 0 0"/>
home_mjc_qpos, home_ros_qpos = format_joint_pos(0, 0, 0.6, 0.1, 0, 0, 0, 0, 0, 0)
# stow manipulator

mjc_qpos, ros_qpos = format_joint_pos(
    arm             = 0,
    gripper         = 0,
    head_pan        = 0,
    head_tilt       = 0,
    lift            = 0.23,
    wrist_pitch     = -0.4,
    wrist_roll      = 0,
    wrist_yaw       = pi,
    base_rotate     = 0,
    base_translate  = 0,
)
node.publish_move_manipulator(home_ros_qpos)

[INFO] [1740903151.533313009] [mujoco_publisher]: Published move_by_pos_cmd: [0.1, 0.0, 0.0, 0.0, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0]


In [45]:
node.destroy_node()
rclpy.shutdown()